<div align="center">

# <span style="color:#2E86C1;"> Lab 1: Data Preparation for Machine Learning</span>

**Author:** [<span style="color:#8E44AD;">Dr. D Bhanu Prakash</span>](https://dbhanuprakash233.github.io)

<img src="https://github.com/dbhanuprakash233/SSSIHL_DBP/blob/main/assets/SssihlLogo.png?raw=true" alt="University Logo" width="80"/>

**<span style="color:#16A085;">Sri Sathya Sai Institute of Higher Learning</span>**  
<span style="color:#5D6D7E;">Prasanthi Nilayam - 515 134, Andhra Pradesh, India.</span>

**Course:** <span style="color:#D35400;">Data Analysis and Visualisation</span>  
**Course Code:** <span style="color:#1ABC9C;">Minor</span>

</div>

### A Hands-On Lab: Data Preparation → Data Cleaning → Data Transformation

**Reference text:** Jason Brownlee, *Data Preparation for Machine Learning*, Machine Learning Mastery
(Also drawing on general practice from Kuhn & Johnson, *Feature Engineering and Selection*, and the scikit-learn documentation.)

---


### Why should you care about this "boring" topic?

Ask any working data scientist how they spend their week, and you'll hear some version of the same joke:

> "80% of the job is cleaning data, and the other 20% is complaining about cleaning data."

It's not really a joke — it's close to the truth. A fancy neural network trained on garbage data will confidently produce garbage predictions. **Garbage In, Garbage Out (GIGO)** is the single most important law in applied machine learning, more important than any algorithm you will learn this semester.

By the end of this lab you will be able to:

1. Explain the difference between **Data Preparation**, **Data Cleaning**, and **Data Transformation**, and where each fits in the ML pipeline.
2. Detect and handle **missing values**, **outliers**, **duplicates**, and **low-information columns**.
3. Apply the major **feature scaling**, **encoding**, **discretization**, and **power transform** techniques correctly.
4. Build a **leak-free** preprocessing pipeline using `scikit-learn`.
5. Demonstrate, with real numbers, that good data preparation **improves model accuracy**.

We will work with the famous **Pima Indians Diabetes Dataset** — a small, messy, real medical dataset that Jason Brownlee uses throughout his book precisely because it has almost every data quality problem you can imagine, hidden in plain sight.



## 0. Setup

Run the cell below first. It imports everything we need for the whole lab.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (MinMaxScaler, StandardScaler, RobustScaler,
                                    OneHotEncoder, OrdinalEncoder, LabelEncoder,
                                    KBinsDiscretizer, PowerTransformer)
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)

print("Setup complete. Libraries loaded.")


## 1. Data Preparation — The Big Picture

**Data Preparation** is the umbrella term for *everything* you do to raw data before it is fit to be fed into a learning algorithm. Brownlee defines a simple, memorable **3-step process**:

| Step | Question it answers | Typical tasks |
|---|---|---|
| **1. Data Selection** | *Which data, and which parts of it, are actually relevant?* | Choosing rows/columns, sampling, deciding what to exclude |
| **2. Data Cleaning** | *Is the data correct and complete?* | Fixing/removing missing values, outliers, duplicates, errors |
| **3. Data Transformation** | *Is the data in the right shape/scale/format for the algorithm?* | Scaling, encoding, discretizing, reducing dimensionality |

A useful mental model:

```
Raw Data  --->  [SELECT]  --->  [CLEAN]  --->  [TRANSFORM]  --->  Model-Ready Data
```

**Why does the *order* matter?** Because each step depends on the one before it. You cannot sensibly scale a column that still contains missing values encoded as `NaN` (the scaler will break or silently propagate `NaN` everywhere). You cannot meaningfully encode a categorical column if duplicate rows are inflating certain categories. Preparation is a *pipeline*, not a checklist you can do in any order.

### A critical rule: avoid *Data Leakage*

> Any statistic used to clean or transform your data (a mean, a min/max, a most-frequent category) must be **learned only from the training set**, then **applied** to the validation/test set. Never compute it on the full dataset before splitting.

This single mistake — computing scaling parameters on the *whole* dataset — is one of the most common bugs in student (and professional!) ML code. We'll build our pipeline the right way in Section 4 using `scikit-learn`'s `Pipeline` object, which handles this automatically.

---

### Meet the dataset: Pima Indians Diabetes

- **Task**: predict whether a patient has diabetes (binary classification).
- **8 numeric input features**: number of pregnancies, plasma glucose, blood pressure, skin fold thickness, insulin, BMI, diabetes pedigree function, age.
- **1 target**: `class` (0 = no diabetes, 1 = diabetes).
- **768 rows.**

This dataset is infamous in the ML teaching world because several columns use **`0` as a stand-in for "missing"** — which is medically impossible (nobody has a blood pressure of 0) but numerically invisible unless you go looking for it. It's the perfect specimen for a data-cleaning autopsy.


In [ ]:

# Brownlee hosts a clean copy of many classic datasets on GitHub -- we load it directly.
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"
columns = ['preg', 'plas', 'pres', 'skin', 'insu', 'mass', 'pedi', 'age', 'class']

try:
    df = pd.read_csv(url, header=None, names=columns)
    print("Loaded dataset directly from source.")
except Exception as e:
    # Fallback: synthesize an equivalent messy dataset locally so the lab still works offline
    print("Network fetch failed, generating a synthetic stand-in dataset instead:", e)
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=768, n_features=8, n_informative=5,
                                n_redundant=1, random_state=42)
    df = pd.DataFrame(X, columns=columns[:-1])
    df = df.abs()  # keep values positive, mimicking medical measurements
    df['class'] = y
    # inject the same "zero means missing" quirk for realism
    for c in ['plas', 'pres', 'skin', 'insu', 'mass']:
        mask = np.random.rand(len(df)) < 0.1
        df.loc[mask, c] = 0

df.shape


In [ ]:

df.head(10)



**🔎 In-class discussion (2 minutes):** Look at the `pres` (blood pressure), `skin` (skinfold thickness), and `insu` (insulin) columns above. Do you notice anything biologically suspicious? Write your observation before scrolling further — we'll confirm it with code in Section 2.


In [ ]:

# Data Selection step: basic structural overview
df.info()
print(df.info())
print()
print("Summary statistics:")
df.describe()



Look closely at the **minimum** values in the table above for `plas`, `pres`, `skin`, `insu`, and `mass`. Several of them are **0**. A blood pressure (`pres`) of 0, or a BMI (`mass`) of 0, is not a real biological measurement — it is almost certainly a **missing value that was encoded as zero** by whoever originally digitized this data. This is a classic "silent" data quality problem: `df.isnull().sum()` will report **zero missing values**, because technically there are no `NaN`s — the problem is hiding as a valid-looking number.


In [ ]:

print("Pandas' isnull() sees NO missing values (a trap!):")
print(df.isnull().sum())



## 2. Data Cleaning

Brownlee organizes data cleaning around four recurring "messy data" problems. We'll tackle each one in turn, on this exact dataset.

1. **Missing values** (including disguised missing values, like our zeros)
2. **Outliers**
3. **Duplicate rows**
4. **Columns with little or no useful information** (e.g. a single constant value)

### 2.1 Missing Values

#### Step A — Mark disguised missing values as real `NaN`s
Domain knowledge tells us that `plas`, `pres`, `skin`, `insu`, and `mass` cannot legitimately be zero. We convert those zeros to proper `NaN` so pandas/scikit-learn can recognize and handle them correctly.


In [ ]:

cols_with_disguised_missing = ['plas', 'pres', 'skin', 'insu', 'mass']

df_clean = df.copy()
df_clean[cols_with_disguised_missing] = df_clean[cols_with_disguised_missing].replace(0, np.nan)

print("Missing values per column, now correctly detected:")
print(df_clean.isnull().sum())
print()
print(f"Total missing cells: {df_clean.isnull().sum().sum()}  "
      f"({100*df_clean.isnull().sum().sum()/df_clean.size:.1f}% of all cells)")



#### Step B — Decide what to do about it: the three classic strategies

| Strategy | What it does | When to use it | Risk |
|---|---|---|---|
| **Listwise deletion** (`dropna` rows) | Remove any row containing a missing value | Missingness is rare (<5%) and random | Throws away real data, can bias the sample |
| **Column deletion** | Drop a whole column | A column is *mostly* missing (e.g. >50%) | Loses a potentially predictive feature |
| **Imputation** | Fill in a plausible estimate (mean, median, mode, KNN, model-based) | Missingness is moderate | Introduces some artificial certainty into the data |

Let's *see* the cost of the naive approach (deleting rows) before choosing imputation.


In [ ]:

rows_before = len(df_clean)
rows_after_dropna = len(df_clean.dropna())
print(f"Rows before dropping: {rows_before}")
print(f"Rows after dropna():  {rows_after_dropna}")
print(f"We would LOSE {rows_before - rows_after_dropna} rows "
      f"({100*(rows_before-rows_after_dropna)/rows_before:.1f}% of the dataset)!")



Losing nearly half the data is a steep price. Instead, we will use **statistical imputation** — replacing each missing value with a summary statistic (mean or median) computed from the *rest of the column*. `scikit-learn`'s `SimpleImputer` does this cleanly, and — importantly — it is a *fittable* transformer, meaning we can fit it only on training data (no leakage) and apply it elsewhere.

**Mean vs. Median?** Use the **median** when a column is skewed or has outliers (median is robust to extreme values); use the **mean** for roughly symmetric, well-behaved distributions. Let's check skewness quickly.


In [ ]:

skewness = df_clean[cols_with_disguised_missing].skew()
print("Skewness of columns with missing values:")
print(skewness)
print("\n(Values far from 0 indicate a skewed distribution -> prefer median imputation)")


In [ ]:

imputer = SimpleImputer(strategy='median')
df_imputed = df_clean.copy()
df_imputed[cols_with_disguised_missing] = imputer.fit_transform(df_clean[cols_with_disguised_missing])

print("Missing values after median imputation:")
print(df_imputed.isnull().sum())



> **A more powerful alternative: `KNNImputer`.** Instead of filling every missing value with a single global statistic, it looks at the *k* most similar rows (nearest neighbours, by the other features) and imputes using their average. This often gives more realistic values, at the cost of more compute. Try it below and compare the imputed `mass` (BMI) values to the simple median approach.


In [ ]:

knn_imputer = KNNImputer(n_neighbors=5)
knn_result = pd.DataFrame(
    knn_imputer.fit_transform(df_clean[cols_with_disguised_missing]),
    columns=cols_with_disguised_missing
)

comparison = pd.DataFrame({
    'original_with_NaN': df_clean['mass'],
    'median_imputed': df_imputed['mass'],
    'knn_imputed': knn_result['mass']
})
comparison[df_clean['mass'].isnull()].head(10)



### 2.2 Outliers

An **outlier** is an observation that lies an abnormal distance from other values. Outliers can be genuine (a very tall person), or they can be data-entry errors (a "height" of 999cm). Either way, they can distort statistics like the mean and can pull linear models badly off course.

Two standard *statistical* detection methods:

- **Standard Deviation method**: flag anything beyond `mean ± k·σ` (commonly *k* = 3, assumes roughly Normal data).
- **Interquartile Range (IQR) method**: flag anything below `Q1 − 1.5·IQR` or above `Q3 + 1.5·IQR`. More robust, doesn't assume normality — this is exactly the rule your box-plot whiskers use.

We'll also peek at a *model-based* method, `IsolationForest`, which detects outliers in multiple dimensions simultaneously (not just one column at a time).


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_imputed[['plas','pres','skin','insu','mass','pedi','age']].boxplot(ax=axes[0], rot=45)
axes[0].set_title("Box plots (IQR outlier rule = whiskers)")

df_imputed['insu'].hist(bins=30, ax=axes[1])
axes[1].set_title("Insulin distribution (notice the long right tail)")
plt.tight_layout()
plt.show()


In [ ]:

def iqr_outlier_bounds(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k*iqr, q3 + k*iqr

for col in ['insu', 'mass', 'pedi']:
    low, high = iqr_outlier_bounds(df_imputed[col])
    n_outliers = ((df_imputed[col] < low) | (df_imputed[col] > high)).sum()
    print(f"{col:6s}: valid range ≈ [{low:.1f}, {high:.1f}]  ->  {n_outliers} outliers flagged")


### Isolation Forest (iForest)

Isolation Forest (iForest) is an **unsupervised machine learning algorithm** designed specifically for **anomaly (outlier) detection**.

Unlike many traditional anomaly detection algorithms that attempt to model normal observations, Isolation Forest works by **isolating observations**.

It was introduced by: Liu, Ting, Zhou (2008)

The key idea is:

> **Anomalies are few and different. Therefore, they are easier to isolate than normal observations.**

Isolation Forest is an ensemble of many random binary trees called **Isolation Trees (iTrees).**

Each tree isolates observations by repeatedly:

- Randomly selecting a feature
- Randomly selecting a split value

Points that become isolated after only a few splits are considered anomalies.

Traditional algorithms learn:

> "What is normal?"

Isolation Forest learns:

> "What is easy to isolate?"

This makes it:

- Faster
- More scalable
- Effective in high-dimensional data

---

### Working Principle

Each Isolation Tree is built as follows:

Step 1: Randomly choose a feature.

Example: Features: Age, Salary . Suppose Age is selected.

Step 2: Randomly choose a split value.

Example: Age values

```
20
22
24
26
80
```

Random split:

```
Age < 40
```

Immediately,

80 goes to one side.

Step 3: Repeat recursively until

- only one sample remains
- or maximum tree depth reached

---

Example dataset

| Person | Age | Salary |
|---------|----:|-------:|
| A | 22 | 25000 |
| B | 25 | 28000 |
| C | 24 | 26000 |
| D | 23 | 27000 |
| E | 75 | 150000 |

Tree

```
                Age < 40
              /          \
         A B C D          E
```

Notice

E is isolated after just one split.

This is an anomaly.


### Advantages

- Fast: O\(number of trees $\times$ subsample size $\times$ log \(subsample size\)\)

where

- t = number of trees
- ψ = subsample size

- Scalable: Works well on millions of observations
- No Distance Calculations unlike LOF, kNN.
- Handles High Dimensions: Can work with hundreds of features.
- No need to assume distribution: Gaussian, Normal, Linear

### Limitations

- Different forests produce slightly different scores.
- May perform poorly if normal observations are already sparse.
- Requires encoding for Categorical Variables.
- Less effective on Small Datasets.

### Important Hyperparameters

- n_estimators: Number of trees: Default value is $100$.
- max_samples: Subsample size: Default value is $256$.
- contamination: Expected fraction of anomalies. Example: $0.05$ means $5\%$ anomalies.
- random_state: for reproducibility.

### Comparison with Other Algorithms

| Algorithm | Supervised | Distance Based | Fast | High-dimensional |
|------------|-----------|---------------|------|------------------|
| Isolation Forest | No | No | Yes | Excellent |
| LOF | No | Yes | Moderate | Moderate |
| One-Class SVM | No | No | Slow | Moderate |
| DBSCAN | No | Yes | Slow | Poor |
| kNN Outlier | No | Yes | Slow | Moderate |

In [ ]:

# Model-based, multivariate outlier detection
iso = IsolationForest(contamination=0.03, random_state=42)
outlier_flags = iso.fit_predict(df_imputed.drop(columns='class'))
print(f"IsolationForest flagged {(outlier_flags == -1).sum()} rows as multivariate outliers "
      f"out of {len(df_imputed)}.")

# In practice you'd inspect these rows, then decide: cap them (winsorize),
# remove them, or leave them (if you believe they're genuine and informative).
df_imputed[outlier_flags == -1].head()



> **⚠️ A word of caution:** never delete outliers automatically without looking at them first. In medical data, an "outlier" patient might be exactly the case your model most needs to learn from. Deleting inconvenient data points to make your model's accuracy look better is a form of data malpractice.

### 2.3 Duplicate Rows

Duplicate rows can arise from bugs in data collection pipelines (e.g., double form submission) and they silently give certain observations extra "voting power" during training.


In [ ]:

n_dupes = df_imputed.duplicated().sum()
print(f"Number of exact duplicate rows: {n_dupes}")

# General pattern (safe to run even if the count is 0):
df_imputed = df_imputed.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df_imputed.shape}")



### 2.4 Columns with Little or No Information

A column that has the **same single value in every row** (zero variance) carries no information at all — it can never help a model distinguish between classes, and some algorithms will even error out on it. More generally, columns with *very low variance* are usually poor predictors and can add noise. `VarianceThreshold` automates this check.


In [ ]:

selector = VarianceThreshold(threshold=0.0)  # threshold=0.0 removes only truly constant columns
selector.fit(df_imputed.drop(columns='class'))

kept_columns = df_imputed.drop(columns='class').columns[selector.get_support()]
dropped_columns = df_imputed.drop(columns='class').columns[~selector.get_support()]

print("Columns kept:  ", list(kept_columns))
print("Columns dropped (zero variance):", list(dropped_columns) if len(dropped_columns) else "None -- all columns carry information")



Our dataset happens to be well-behaved here (no constant columns), but on real-world tabular data — especially wide datasets with hundreds of sensor or log-derived columns — this check regularly saves you from feeding the model useless noise.

### ✅ Checkpoint: Cleaning summary

| Problem | Technique used | Result |
|---|---|---|
| Disguised missing values (zeros) | Domain-knowledge replace → `NaN` | 5 columns flagged |
| Missing values | Median / KNN imputation | 0 missing values remain |
| Outliers | IQR + IsolationForest | Identified, inspected (not blindly deleted) |
| Duplicate rows | `drop_duplicates()` | Removed |
| Zero-variance columns | `VarianceThreshold` | None found in this dataset |

`df_imputed` is now our **clean** dataset. Next, we transform it into a shape better suited for learning algorithms.



## 3. Data Transformation

Cleaning fixes *correctness*. Transformation fixes *form* — reshaping features into representations that algorithms can exploit efficiently. We'll cover the four transformation families most likely to appear on your exam and in real projects:

1. **Feature Scaling** (Normalization / Standardization / Robust scaling)
2. **Categorical Encoding** (Ordinal / One-Hot)
3. **Discretization** (turning continuous numbers into bins/categories)
4. **Power Transforms** (fixing skewed distributions) and a peek at **Dimensionality Reduction**

### 3.1 Feature Scaling — Why it matters

Look at the raw ranges in our dataset: `preg` ranges roughly 0–17, while `insu` ranges up to ~850. Distance-based algorithms (k-NN, k-Means, SVMs) and gradient-descent-based algorithms (logistic regression, neural nets) are all sensitive to feature scale — a feature with a numerically larger range will dominate the distance/gradient calculations for no good statistical reason. **Tree-based models (Decision Trees, Random Forests, Gradient Boosting) are the notable exception** — they split on thresholds per feature independently, so they don't need scaling.

| Technique | Formula | Resulting range | Sensitive to outliers? |
|---|---|---|---|
| **Normalization** (`MinMaxScaler`) | (x − min) / (max − min) | exactly [0, 1] | Yes — a single extreme value stretches the whole scale |
| **Standardization** (`StandardScaler`) | (x − mean) / std | mean 0, std 1, unbounded | Somewhat |
| **Robust Scaling** (`RobustScaler`) | (x − median) / IQR | centered on median, unbounded | No — designed for exactly this |


In [ ]:

feature_cols = [c for c in df_imputed.columns if c != 'class']
X_demo = df_imputed[feature_cols].copy()

scalers = {
    'Original': X_demo['insu'],
    'MinMax [0,1]': pd.Series(MinMaxScaler().fit_transform(X_demo[['insu']]).ravel()),
    'Standardized (z-score)': pd.Series(StandardScaler().fit_transform(X_demo[['insu']]).ravel()),
    'Robust': pd.Series(RobustScaler().fit_transform(X_demo[['insu']]).ravel()),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, (name, series) in zip(axes, scalers.items()):
    ax.hist(series, bins=30, color='steelblue')
    ax.set_title(name, fontsize=10)
plt.suptitle("Same 'insu' column, four different scales (shape is preserved, scale is not!)")
plt.tight_layout()
plt.show()



**Key insight for the exam:** scaling changes the *numbers*, never the *shape* of the distribution. If a feature is skewed before scaling, it is equally skewed after scaling — scaling is not a fix for skewness (that's what Power Transforms, Section 3.4, are for).

### 3.2 Encoding Categorical Variables

Our Pima dataset happens to be all-numeric, so let's construct a small synthetic categorical column to demonstrate encoding — a skill you'll need constantly on other datasets (survey data, product categories, etc).

- **Ordinal Encoding**: maps categories to integers `0, 1, 2, ...`. Only appropriate when categories have a **genuine order** (e.g. `Low < Medium < High`).
- **One-Hot Encoding**: creates a separate binary (0/1) column per category. Appropriate for **nominal** (unordered) categories, and required by most linear/distance-based models — otherwise they would wrongly assume an order exists.


In [ ]:

# Simulate a plausible categorical feature: BMI category (a common clinical derived feature)
def bmi_category(bmi):
    if bmi < 18.5:  return 'Underweight'
    if bmi < 25:    return 'Normal'
    if bmi < 30:    return 'Overweight'
    return 'Obese'

df_imputed['bmi_category'] = df_imputed['mass'].apply(bmi_category)
print(df_imputed['bmi_category'].value_counts())


In [ ]:

# Ordinal encoding -- correct here, because the categories DO have a natural order
order = [['Underweight', 'Normal', 'Overweight', 'Obese']]
ordinal_enc = OrdinalEncoder(categories=order)
df_imputed['bmi_category_ordinal'] = ordinal_enc.fit_transform(df_imputed[['bmi_category']])

# One-Hot encoding -- also valid, and safer if you're not sure a model can exploit the ordering
onehot_enc = OneHotEncoder(sparse_output=False, dtype=int)
onehot_cols = onehot_enc.fit_transform(df_imputed[['bmi_category']])
onehot_df = pd.DataFrame(onehot_cols, columns=onehot_enc.get_feature_names_out(['bmi_category']))

display_df = pd.concat([df_imputed[['bmi_category', 'bmi_category_ordinal']].reset_index(drop=True),
                         onehot_df], axis=1)
display_df.head(8)



> **Pitfall to avoid:** never feed a *nominal* (unordered) category through a plain `OrdinalEncoder`/`LabelEncoder` into a linear or distance-based model. If `Red=0, Green=1, Blue=2`, the model will assume `Blue` is somehow "more" than `Red` — a meaningless and misleading relationship that a One-Hot encoding avoids entirely. (`LabelEncoder` is reserved specifically for encoding the **target/label** column in classification, not input features.)

### 3.3 Discretization (Binning)

Discretization converts a continuous variable into discrete buckets. This can help simple models capture non-linear relationships (e.g., risk might jump sharply after age 45, not increase smoothly), and can make patterns easier for humans to interpret in reports and dashboards.


In [ ]:

discretizer = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
df_imputed['age_bin'] = discretizer.fit_transform(df_imputed[['age']]).astype(int)

print("Bin edges chosen (quantile strategy -> roughly equal COUNT per bin):")
print(np.round(discretizer.bin_edges_[0], 1))
print()
print(df_imputed[['age', 'age_bin']].sample(8, random_state=1))



`strategy='quantile'` creates bins with roughly equal numbers of observations; `strategy='uniform'` creates bins of equal width regardless of how many points fall in each; `strategy='kmeans'` clusters values into bins based on similarity. Choose based on whether you care about balanced bin population, balanced bin range, or natural groupings in the data.

### 3.4 Power Transforms — Fixing Skewed Distributions

Many statistical and linear models assume features are roughly **Normally distributed** (bell-curve shaped). Our `insu` (insulin) and `pedi` (pedigree function) columns are heavily right-skewed, as you saw in the histogram earlier. **Power transforms** apply a mathematical function that pulls in the long tail and makes the distribution more symmetric.

- **Box-Cox**: powerful, but requires strictly **positive** data.
- **Yeo-Johnson**: a modern generalization that also works with zero/negative values — safer default choice.


In [ ]:

pt = PowerTransformer(method='yeo-johnson')
insu_transformed = pt.fit_transform(df_imputed[['insu']])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(df_imputed['insu'], bins=30, color='indianred')
axes[0].set_title(f"Before  (skew = {df_imputed['insu'].skew():.2f})")
axes[1].hist(insu_transformed, bins=30, color='seagreen')
axes[1].set_title(f"After Yeo-Johnson  (skew = {pd.Series(insu_transformed.ravel()).skew():.2f})")
plt.suptitle("Power Transform pulls in the long right tail")
plt.tight_layout()
plt.show()



### 3.5 (Bonus) Dimensionality Reduction with PCA

When you have *many* correlated features, **Principal Component Analysis (PCA)** re-expresses the data using a smaller number of new "synthetic" features (principal components) that capture most of the original variance. This can speed up training, reduce overfitting, and enable visualization of high-dimensional data in 2D.

**Important:** PCA is sensitive to scale, so you must **always standardize your features before applying PCA** — otherwise the component with the numerically largest raw range will dominate.


In [ ]:

X_scaled = StandardScaler().fit_transform(df_imputed[feature_cols])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df_imputed['class'], cmap='coolwarm', alpha=0.6, edgecolor='k', linewidth=0.2)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance explained)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance explained)")
plt.title("8 features compressed into 2 principal components\n(color = diabetes class)")
plt.colorbar(label='class')
plt.show()

print(f"Together, PC1 + PC2 explain {pca.explained_variance_ratio_.sum()*100:.1f}% "
      f"of the total variance in the original 8 features.")



## 4. Putting It All Together: A Leak-Free `Pipeline`

So far we transformed the *whole* dataset at once, which was great for teaching but is **methodologically wrong** for a real project — remember the leakage warning in Section 1! The imputer's median, the scaler's mean/std, etc. must be learned **only** from the training split.

`scikit-learn`'s `Pipeline` + `ColumnTransformer` bundle every step (impute → scale → encode → model) into a single object that:

- Fits all preprocessing statistics **only on the training fold** during `.fit()`
- Applies the *same, frozen* transformation to any new data during `.predict()`
- Makes cross-validation, grid search, and deployment dramatically less error-prone

Let's build one, and — most importantly — let's **prove with numbers** that proper cleaning and transformation improves predictive performance.


In [ ]:

# Start again from the RAW data to build an honest, end-to-end pipeline
X = df.drop(columns='class').copy()
y = df['class'].copy()

# Re-inject the "0 = missing" domain knowledge as an explicit preprocessing step
X[cols_with_disguised_missing] = X[cols_with_disguised_missing].replace(0, np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# --- Pipeline A: NO cleaning/transformation at all (just fill NaN with 0, the naive approach) ---
naive_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0)),
    ('model', LogisticRegression(max_iter=1000))
])

# --- Pipeline B: proper median imputation + standard scaling ---
good_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

naive_scores = cross_val_score(naive_pipeline, X_train, y_train, cv=cv, scoring='accuracy')
good_scores  = cross_val_score(good_pipeline,  X_train, y_train, cv=cv, scoring='accuracy')

print(f"Naive pipeline   (fill 0, no scaling):  {naive_scores.mean():.4f}  (+/- {naive_scores.std():.4f})")
print(f"Cleaned pipeline (median impute+scale): {good_scores.mean():.4f}  (+/- {good_scores.std():.4f})")
print(f"\nImprovement from proper data preparation: {(good_scores.mean()-naive_scores.mean())*100:+.2f} percentage points")



Run the cell above and discuss: even on a small, simple dataset with a simple linear model, correcting the "0 means missing" issue and scaling the features changes cross-validated accuracy. On larger, messier, real-world datasets — and with algorithms more sensitive to scale and noise — the gap from good data preparation is often the difference between a mediocre model and a genuinely useful one.

### A full pipeline with mixed column types

Real datasets almost always mix numeric and categorical columns. `ColumnTransformer` lets you route different columns through different preprocessing steps, all inside one fittable object.


In [ ]:

numeric_features = feature_cols  # all 8 numeric columns
categorical_features = []        # (none native to this dataset, shown here for the general pattern)

numeric_transformer = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

categorical_transformer = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    # ('cat', categorical_transformer, categorical_features),  # uncomment when categorical cols exist
])

full_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

full_pipeline.fit(X_train, y_train)
test_accuracy = full_pipeline.score(X_test, y_test)
print(f"Held-out test accuracy of the full, leak-free pipeline: {test_accuracy:.4f}")



This `full_pipeline` object is now genuinely deployable: hand it a brand-new, raw patient record — zeros, missing values, and all — and it will clean, transform, and predict correctly, using only the statistics it learned from the training data.



## 5. Summary Cheat-Sheet

| Stage | Problem | Tool(s) |
|---|---|---|
| **Preparation** | Understand structure, select relevant data | `.info()`, `.describe()`, `.shape` |
| **Cleaning** | Missing values | `.replace()`, `SimpleImputer`, `KNNImputer` |
| **Cleaning** | Outliers | IQR rule, z-score rule, `IsolationForest` |
| **Cleaning** | Duplicates | `.duplicated()`, `.drop_duplicates()` |
| **Cleaning** | Uninformative columns | `VarianceThreshold` |
| **Transformation** | Different scales | `MinMaxScaler`, `StandardScaler`, `RobustScaler` |
| **Transformation** | Categorical variables | `OrdinalEncoder`, `OneHotEncoder` |
| **Transformation** | Continuous → categorical | `KBinsDiscretizer` |
| **Transformation** | Skewed distributions | `PowerTransformer` (Box-Cox / Yeo-Johnson) |
| **Transformation** | Too many correlated features | `PCA` |
| **Assembly** | Avoiding data leakage | `Pipeline`, `ColumnTransformer` |

---

## 6. In-Class / Homework Exercises

Work through these individually or in pairs. Add your code and answers in new cells below.

1. **Detective work.** For the `skin` (skinfold thickness) column, use the IQR method to count outliers *before* and *after* median imputation of the missing values. Does imputation create or hide any new outliers?
2. **Strategy comparison.** Repeat the Section 4 experiment, but compare **three** pipelines: (a) drop rows with any missing value, (b) mean imputation, (c) KNN imputation. Which gives the best cross-validated accuracy? Can you explain *why*, referencing the skewness numbers from Section 2?
3. **Scaling matters for some models, not others.** Train a `DecisionTreeClassifier` on the *unscaled* vs. *scaled* features. Confirm (and explain in 2–3 sentences) why the accuracy barely changes — unlike what we saw for `LogisticRegression`.
4. **Build your own categorical feature.** Create a `glucose_risk` category from the `plas` column (e.g., Normal / Prediabetic / Diabetic ranges, look up real clinical thresholds). One-Hot encode it and add it to the `full_pipeline`'s `ColumnTransformer`. Does test accuracy improve?
5. **Challenge.** Try `PowerTransformer(method='box-cox')` instead of Yeo-Johnson on the `insu` column. It will raise an error — read the error message carefully and explain *in your own words* why Box-Cox cannot be used here without an extra preprocessing step.

---

## 7. References & Further Reading

- Brownlee, J. *Data Preparation for Machine Learning.* Machine Learning Mastery.
- Brownlee, J. *Pima Indians Diabetes Dataset* — `https://github.com/jbrownlee/Datasets`
- Kuhn, M. & Johnson, K. *Feature Engineering and Selection: A Practical Approach for Predictive Models.*
- scikit-learn User Guide: [Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html), [Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html)